In [58]:
import regex as re

In [50]:
with open('data.txt', 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()

In [51]:
len(text)

173898

In [61]:
class Tokenizer:

    def __init__(self):
        self.merges = {}
        self.vocab = {}

    def get_status(self, tokens):
        pair_count = {}
        for t1, t2 in zip(tokens, tokens[1:]):
            pair_count[(t1, t2)] = pair_count.get((t1,t2), 0) + 1
        return pair_count
    
    def merge(self, tokens, max_pair, new_index):
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == max_pair[0] and tokens[i+1] == max_pair[1]:
                new_tokens.append(new_index)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens
    
    def apply_regrex(self, text):
        gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s\p{L}\p{N}+|\s+(?!\S)|s+""")
        tokens = re.findall(gpt2pat, text)
        tokens = " ".join(tokens)
        return tokens
    
    def train(self, text, new_vocab_size):
        text = self.apply_regrex(text)
        tokens = list(text.encode("utf-8"))
        self.merges = {}
        for i in range (new_vocab_size - 256):
            stats = self.get_status(tokens)
            max_pair = max(stats, key=stats.get)
            new_index = i + 256
            self.merges[max_pair] = new_index
            tokens = self.merge(tokens, max_pair, new_index)
            print(f"Training done for epoch {i}")

        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        for (po, p1), idx in self.merges.items():
            self.vocab[idx] = self.vocab[po] + self.vocab[p1]

    def encode(self, text):
        text = self.apply_regrex(text)
        tokens = list(text.encode("utf-8"))

        for pair, value in self.merges.items():
            encoded_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    encoded_tokens.append(value)
                    i += 2
                else:
                    encoded_tokens.append(tokens[i])
                    i += 1
            tokens = encoded_tokens
        return tokens
    

    def decode(self, tokens):
        tokens = b''.join(self.vocab[token] for token in tokens)
        text = tokens.decode("utf-8", errors="replace")
        return text

In [62]:
tokenizer1 = Tokenizer()
tokenizer1.train(text, new_vocab_size=56000)

Training done for epoch 0
Training done for epoch 1
Training done for epoch 2
Training done for epoch 3
Training done for epoch 4
Training done for epoch 5
Training done for epoch 6
Training done for epoch 7
Training done for epoch 8
Training done for epoch 9
Training done for epoch 10
Training done for epoch 11
Training done for epoch 12
Training done for epoch 13
Training done for epoch 14
Training done for epoch 15
Training done for epoch 16
Training done for epoch 17
Training done for epoch 18
Training done for epoch 19
Training done for epoch 20
Training done for epoch 21
Training done for epoch 22
Training done for epoch 23
Training done for epoch 24
Training done for epoch 25
Training done for epoch 26
Training done for epoch 27
Training done for epoch 28
Training done for epoch 29
Training done for epoch 30
Training done for epoch 31
Training done for epoch 32
Training done for epoch 33
Training done for epoch 34
Training done for epoch 35
Training done for epoch 36
Training do

ValueError: max() iterable argument is empty

In [ ]:
print(tokenizer1.decode(tokenizer1.encode("hello worldjgmhnfdgfb")))

hello worldjgmhnfdgfb


In [ ]:
x= tokenizer1.encode("hello worldjgmhnfdgfb")
y = tokenizer1.decode(x)
print(x,y)

[104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 106, 103, 109, 104, 110, 102, 100, 103, 102, 98] hello worldjgmhnfdgfb
